# Function Testing Notebook

Author: Pete King

This notebook tests custom functions developed in the various helper modules to verify proper operation.

In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import yfinance as yf

import data_prep as dp

DATA_FILENAME='etf_raw_data.csv'
ETF='SPY'

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Import and inspect ETF price data

In [2]:
df = pd.read_csv(
    DATA_FILENAME,
    index_col='date',
    parse_dates=True
)
df

,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.175377,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.347336,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.398914,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.656820,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.760002,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-20,91.570000,73.169998,413.380005,78.919998,94.879997,242.220001,107.849998,582.059998,648.570007,110.220001,...,46.980000,59.310001,49.080002,161.669998,135.289993,81.290001,40.590000,44.650002,145.330002,107.739998
2026-03-23,91.580002,73.449997,404.040009,79.440002,95.180000,247.449997,108.559998,588.000000,655.380005,110.190002,...,47.549999,59.630001,49.270000,163.050003,136.949997,81.180000,40.619999,44.779999,144.770004,110.120003
2026-03-24,91.589996,73.260002,404.130005,79.169998,94.860001,248.779999,108.339996,583.979980,653.179993,109.830002,...,48.450001,60.840000,49.279999,164.000000,136.149994,81.110001,40.290001,45.090000,144.789993,109.680000


## Compute and display daily return

Here we test the ability of the log_return function to compute daily returns and inspect the results.

In [3]:
test_df = df[[ETF]]
etf_return = test_df['SPY'].rolling(2).apply(dp.log_return, raw=True)
test_df[ETF + '_return'] = etf_return.values
test_df

,SPY,SPY_return
date,,
1993-01-29,24.175377,NaN
1993-02-01,24.347336,0.007088
1993-02-02,24.398914,0.002116
1993-02-03,24.656820,0.010515
1993-02-04,24.760002,0.004176
...,...,...
2026-03-20,648.570007,-0.014440
2026-03-23,655.380005,0.010445
2026-03-24,653.179993,-0.003362


In [4]:
chart = alt.Chart(test_df.dropna().reset_index()).mark_circle(size=10).encode(
    x='date:T',
    y=ETF + '_return:Q'
)
chart.properties(height=200, width=800)

alt.Chart(...)

## Discussion

From the chart we can see that the mean of daily returns appears to be nearly zero -- an empirical justification of the zero mean assumption for expected return (E\[R\]).

Since volatility for an asset is a measure of the deviation of returns from expected return, we can get a feel for an asset's volatility just by inspecting the plot.  We see a general trend of baseline low volatility (for example, from Jan 2004 to Jan 2007), with periods of high volatility that tend to gradually revert to baseline (for example during the 'Great Recession', from late 2007, spiking in late 2008 / early 2009, and gradually reverting to a lower baseline by roughly 2012).

In [5]:
etf_return.describe()

count    8344.000000
mean        0.000396
std         0.011721
min        -0.115887
25%        -0.004337
50%         0.000679
75%         0.005916
max         0.135577
Name: SPY, dtype: float64

The main idea with this project is to think of the daily return for each asset as a random variable (R), and then investigate its statistical properties.  

***Right away, from this simple statistical description (above), we can get an idea of what to expect for the properties of an asset's returns (R):***
 - Estimated **expected return** (E\[R\]): 0.04 percent (very close to zero)
 - Estimated long-term (baseline) **volatility**: 1.17 percent

*Note that the financial term "volatility" can have mean interpretations, but here we mean the long-term standard deviation of return (R), assuming zero mean.*

In [6]:
# Compute using the zero-mean assumption for expected return
vol = np.sqrt(
    np.sum(etf_return.dropna().values**2) / len(etf_return.dropna())
)
print(f'Estimated long-term volatility with zero-mean assumption: \
        {vol * 100:2.2f} percent'
     )

Estimated long-term volatility with zero-mean assumption:         1.17 percent
